HEXAPAWN

In [10]:
import time
import ipywidgets as widgets
from IPython.display import display
from games import Game, GameState, alpha_beta_player

In [11]:
class Hexapawn(Game):

    def __init__(self):
        board = {
            (0, 0): 'B', (0, 1): 'B', (0, 2): 'B',
            (2, 0): 'W', (2, 1): 'W', (2, 2): 'W'
        }

        self.initial = GameState(
            'W', 0, board,
            self.get_moves(board, 'W')
        )

    def actions(self, state):
        return state.moves

    def get_moves(self, board, player):
        moves = []
        d = -1 if player == 'W' else 1

        for (r, c), p in board.items():
            if p != player:
                continue

            # เดินตรง
            pos = (r + d, c)
            if 0 <= pos[0] < 3 and pos not in board:
                moves.append(((r, c), pos))

            # กินเฉียง
            for dc in [-1, 1]:
                pos = (r + d, c + dc)
                if (0 <= pos[0] < 3 and
                    0 <= pos[1] < 3 and
                    pos in board and
                    board[pos] != player):
                    moves.append(((r, c), pos))

        return moves

    def result(self, state, move):
        board = state.board.copy()
        a, b = move

        piece = board.pop(a)
        board[b] = piece

        player = 'B' if state.to_move == 'W' else 'W'

        if board.get((0, 0)) == 'W' or \
           board.get((0, 1)) == 'W' or \
           board.get((0, 2)) == 'W':
            utility = 1
        elif board.get((2, 0)) == 'B' or \
             board.get((2, 1)) == 'B' or \
             board.get((2, 2)) == 'B':
            utility = -1
        else:
            utility = 0

        return GameState(
            player,
            utility,
            board,
            self.get_moves(board, player)
        )

    def utility(self, state, player):
        return state.utility if player == 'W' else -state.utility

    def terminal_test(self, state):
        return state.utility != 0 or len(state.moves) == 0


In [ ]:
class InteractiveHexapawn:

    def __init__(self):
        self.game = Hexapawn()
        self.state = self.game.initial
        self.selected = None
        self.last_board = {}     
        self.setup_game()

    def setup_game(self):

        self.mode = widgets.Dropdown(
            options=['Human vs AlphaBeta', 'AlphaBeta vs AlphaBeta'],
            value='Human vs AlphaBeta',
            description='Mode:'
        )
        self.mode.observe(self.new_game, names='value')

        self.status = widgets.HTML(value="<h3>White's turn</h3>")

        self.buttons = [[None]*3 for _ in range(3)]
        for r in range(3):
            for c in range(3):
                self.buttons[r][c] = self._new_button(r, c)

        reset = widgets.Button(description='New Game', button_style='success')
        reset.on_click(self.new_game)

        board = [widgets.HBox(self.buttons[r]) for r in range(3)]

        self.ui = widgets.VBox([
            widgets.HTML("<h2>Hexapawn vs AlphaBeta AI</h2>"),
            self.mode,
            self.status,
            widgets.VBox(board),
            reset
        ])

        self.update(force_all=True)

    def _new_button(self, r, c):
        """สร้าง Button ตัวใหม่เปล่าๆ พร้อม handler (ยังไม่ตั้งสี/label)"""
        btn = widgets.Button(
            description='',
            layout=widgets.Layout(width='90px', height='90px')
        )
        btn.row = r
        btn.col = c
        btn.on_click(self.click)
        return btn

    def _style_button(self, btn, pos, piece):
        if pos == self.selected:
            bg, fg = '#facc15', '#000000'
        elif self.selected is not None and (self.selected, pos) in self.state.moves:
            bg, fg = '#22c55e', '#000000'
        elif piece == 'W':
            bg, fg = '#000000', '#ffffff'
        elif piece == 'B':
            bg, fg = '#ffffff', '#000000'
        else:
            bg, fg = '#808080', '#000000'

        btn.description = piece if piece else ''
        btn.style = widgets.ButtonStyle(
            button_color=bg, text_color=fg, font_weight='bold'
        )

    def click(self, button):

        if self.game.terminal_test(self.state):
            return
        if self.mode.value == 'Human vs AlphaBeta' and self.state.to_move != 'W':
            return

        pos = (button.row, button.col)

        if self.selected is None:
            if self.state.board.get(pos) == self.state.to_move:
                self.selected = pos
                self.update()
            return

        move = (self.selected, pos)

        if move in self.state.moves:
            self.state = self.game.result(self.state, move)
            self.selected = None
            self.update()

            if (self.mode.value == 'Human vs AlphaBeta'
                and self.state.to_move == 'B'
                and not self.game.terminal_test(self.state)):
                self.ai_move()
        else:
            if self.state.board.get(pos) == self.state.to_move:
                self.selected = pos
            else:
                self.selected = None
            self.update()

    def ai_move(self):
        time.sleep(0.3)

        if self.game.terminal_test(self.state):
            self.update()
            return

        move = alpha_beta_player(self.game, self.state)
        self.state = self.game.result(self.state, move)
        self.update()

        if (self.mode.value == 'AlphaBeta vs AlphaBeta'
            and not self.game.terminal_test(self.state)):
            self.ai_move()

    def update(self, force_all=False):

        cur_board = self.state.board

        changed = {
            pos for pos in set(cur_board) | set(self.last_board)
            if cur_board.get(pos) != self.last_board.get(pos)
        }

        for r in range(3):
            for c in range(3):
                pos = (r, c)
                piece = cur_board.get(pos)
                need_recreate = force_all or pos in changed

                if need_recreate:
                    new_btn = self._new_button(r, c)
                    self._style_button(new_btn, pos, piece)
                    self.buttons[r][c] = new_btn
                else:
                    self._style_button(self.buttons[r][c], pos, piece)

        self.ui.children[3].children = [
            widgets.HBox(self.buttons[r]) for r in range(3)
        ]

        self.last_board = dict(cur_board)

        if self.game.terminal_test(self.state):
            if self.state.utility == 1:
                self.status.value = "<h3>White Wins!</h3>"
            elif self.state.utility == -1:
                self.status.value = "<h3>Black Wins!</h3>"
            else:
                self.status.value = "<h3>Draw!</h3>"
        else:
            self.status.value = f"<h3>{'White' if self.state.to_move=='W' else 'Black'}'s turn</h3>"

    def new_game(self, change=None):
        self.state = self.game.initial
        self.selected = None
        self.last_board = {}
        self.update(force_all=True)

        if self.mode.value == 'AlphaBeta vs AlphaBeta':
            self.ai_move()

    def display(self):
        display(self.ui)


In [13]:
game_ui = InteractiveHexapawn()
game_ui.display()